# Crop Yield Prediction

**Tabular Regression Project**

Models: CatBoost · LightGBM · XGBoost
Baselines: LazyPredict · FLAML AutoML
Target: `yield_tons_per_ha`

## 2 · Project Overview

We predict **crop yield** (tons per hectare) based on crop type, weather (rainfall, temperature), soil properties (pH, N/P/K), irrigation method, fertilizer usage, and pesticide level.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Perform EDA on a tabular regression dataset.
2. Encode categorical features for tree-based and linear models.
3. Build a baseline Linear Regression model.
4. Use LazyPredict for rapid model benchmarking.
5. Run FLAML AutoML for automated model selection.
6. Train CatBoost, LightGBM, XGBoost with GPU→CPU fallback.
7. Evaluate with RMSE, MAE, R², and residual analysis.

## 4 · Problem Statement

Given a field's crop type, weather conditions, soil chemistry, irrigation, fertilizer, and pesticide usage, predict the expected yield in tons per hectare.

## 5 · Why This Project Matters

- **Crop yield prediction** is critical for food security and agricultural planning.
- Understanding yield drivers enables precision agriculture.
- Climate adaptation requires quantifying temperature and rainfall effects.
- Demonstrates regression with agricultural science-informed features.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Rows** | 7,500 |
| **Features** | crop_type, rainfall_mm, avg_temp_c, soil_ph, soil_nitrogen, soil_phosphorus, soil_potassium, irrigation, fertilizer_kg_per_ha, pesticide_usage, field_area_ha |
| **Target** | yield_tons_per_ha (continuous) |
| **Range** | ~0.2 – 15 tons/ha |

## 7 · Dataset Source and License Notes

- **Source**: Synthetic dataset generated for educational purposes.
- **License**: Public / educational use. No PII.
- **Limitations**: Simplified patterns — real-world data would have more features and noise.

## 8 · Environment Setup

Install any missing packages. GPU is auto-detected with CPU fallback.

In [ ]:
import subprocess, sys

def _install(pkg):
    try:
        __import__(pkg)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for pkg in ["catboost", "lightgbm", "xgboost", "flaml", "lazypredict"]:
    _install(pkg)

print("All packages ready.")

## 9 · Imports

In [ ]:
import os, json, time, warnings, random, gc
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OrdinalEncoder
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score, root_mean_squared_error

warnings.filterwarnings("ignore")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
print("Imports complete.")

## 10 · Configuration / Constants

In [ ]:
TARGET = "yield_tons_per_ha"
SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
print(f"Target: {TARGET}")
print(f"Save dir: {SAVE_DIR}")

## 11 · Dataset Download or Loading

Synthetic dataset: 7,500 agricultural field records with soil, weather, and farming features.

In [ ]:
np.random.seed(SEED)
n = 7500
crop_type = np.random.choice(["Wheat", "Rice", "Corn", "Soybean", "Cotton"], n,
                              p=[0.25, 0.2, 0.25, 0.15, 0.15])
crop_base = {"Wheat": 3.5, "Rice": 4.5, "Corn": 5.5, "Soybean": 2.8, "Cotton": 2.2}
crop_val = np.array([crop_base[c] for c in crop_type])

rainfall_mm = np.round(np.random.normal(800, 250, n).clip(100, 2000), 0).astype(int)
avg_temp_c = np.round(np.random.normal(22, 5, n).clip(5, 40), 1)
soil_ph = np.round(np.random.normal(6.5, 0.8, n).clip(4, 9), 1)
soil_nitrogen = np.round(np.random.normal(150, 50, n).clip(20, 400), 0).astype(int)
soil_phosphorus = np.round(np.random.normal(30, 10, n).clip(5, 80), 0).astype(int)
soil_potassium = np.round(np.random.normal(200, 60, n).clip(50, 500), 0).astype(int)
irrigation = np.random.choice(["Rainfed", "Drip", "Sprinkler", "Flood"], n,
                               p=[0.3, 0.25, 0.25, 0.2])
irr_mult = {"Rainfed": 0.75, "Drip": 1.15, "Sprinkler": 1.05, "Flood": 1.0}
irr_val = np.array([irr_mult[i] for i in irrigation])

fertilizer_kg_per_ha = np.round(np.random.lognormal(4.5, 0.5, n).clip(20, 500), 0).astype(int)
pesticide_usage = np.random.choice(["None", "Low", "Medium", "High"], n,
                                    p=[0.1, 0.3, 0.35, 0.25])
pest_mult = {"None": 0.8, "Low": 0.9, "Medium": 1.0, "High": 1.05}
pest_val = np.array([pest_mult[p] for p in pesticide_usage])
field_area_ha = np.round(np.random.lognormal(2, 0.8, n).clip(0.5, 200), 1)

# Yield model
temp_factor = 1 - 0.01 * np.abs(avg_temp_c - 25) ** 1.5
rain_factor = np.minimum(rainfall_mm / 700, 1.3)
ph_factor = 1 - 0.05 * np.abs(soil_ph - 6.5)
nutrient_factor = 0.001 * soil_nitrogen + 0.002 * soil_phosphorus + 0.0005 * soil_potassium
fert_factor = 0.002 * np.minimum(fertilizer_kg_per_ha, 300)

yield_tons_per_ha = crop_val * temp_factor * rain_factor * ph_factor
yield_tons_per_ha = yield_tons_per_ha * irr_val * pest_val + nutrient_factor + fert_factor
yield_tons_per_ha = np.round(yield_tons_per_ha + np.random.normal(0, 0.4, n), 2).clip(0.2, 15)

df = pd.DataFrame({
    "crop_type": crop_type, "rainfall_mm": rainfall_mm, "avg_temp_c": avg_temp_c,
    "soil_ph": soil_ph, "soil_nitrogen": soil_nitrogen, "soil_phosphorus": soil_phosphorus,
    "soil_potassium": soil_potassium, "irrigation": irrigation,
    "fertilizer_kg_per_ha": fertilizer_kg_per_ha, "pesticide_usage": pesticide_usage,
    "field_area_ha": field_area_ha, "yield_tons_per_ha": yield_tons_per_ha
})
print(f"Dataset shape: {df.shape}")
print(f"Target stats:\n{df['yield_tons_per_ha'].describe()}")
df.head()

## 12 · Data Validation Checks

Verify the dataset is complete and correctly structured.

In [ ]:
print("=" * 50)
print("DATA VALIDATION")
print("=" * 50)
print(f"\nShape: {df.shape}")
print(f"\nMissing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nDtypes:\n{df.dtypes}")
assert TARGET in df.columns, f"Target '{TARGET}' not found!"
print(f"\nTarget '{TARGET}' confirmed.")
print(f"\nTarget stats:\n{df[TARGET].describe()}")

## 13 · Exploratory Data Analysis

Explore feature distributions, correlations, and relationships.

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0][0].scatter(df["rainfall_mm"], df[TARGET], alpha=0.2, s=10)
axes[0][0].set_xlabel("Rainfall (mm)"); axes[0][0].set_ylabel("Yield (t/ha)")
axes[0][0].set_title("Rainfall vs Yield")

axes[0][1].scatter(df["avg_temp_c"], df[TARGET], alpha=0.2, s=10)
axes[0][1].set_xlabel("Avg Temperature (°C)"); axes[0][1].set_ylabel("Yield (t/ha)")
axes[0][1].set_title("Temperature vs Yield")

axes[0][2].scatter(df["soil_nitrogen"], df[TARGET], alpha=0.2, s=10)
axes[0][2].set_xlabel("Soil Nitrogen"); axes[0][2].set_ylabel("Yield (t/ha)")
axes[0][2].set_title("Nitrogen vs Yield")

df.boxplot(column=TARGET, by="crop_type", ax=axes[1][0])
axes[1][0].set_title("Yield by Crop Type")
axes[1][0].tick_params(axis="x", rotation=45)

df.boxplot(column=TARGET, by="irrigation", ax=axes[1][1])
axes[1][1].set_title("Yield by Irrigation")
axes[1][1].tick_params(axis="x", rotation=45)

corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0, ax=axes[1][2], square=True, cbar_kws={"shrink": 0.8})
axes[1][2].set_title("Correlation")

plt.tight_layout()
plt.show()

## 14 · Target Analysis

Examine the distribution of `yield_tons_per_ha`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

axes[0].hist(df[TARGET], bins=30, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].set_title(f"Distribution of {TARGET}")
axes[0].set_xlabel("Yield (tons/ha)")

df.boxplot(column=TARGET, by="crop_type", ax=axes[1])
axes[1].set_title("Yield by Crop Type")
axes[1].tick_params(axis="x", rotation=45)

plt.tight_layout()
plt.show()
print(f"Range: {df[TARGET].min():.2f} to {df[TARGET].max():.2f} tons/ha")
print(f"Mean: {df[TARGET].mean():.2f}, Median: {df[TARGET].median():.2f}")

## 15 · Train/Validation/Test Split Strategy

80/20 train/test split.

In [ ]:
X = df.drop(columns=[TARGET])
y = df[TARGET]

# Encode categorical features
cat_cols = X.select_dtypes(include="object").columns.tolist()
if cat_cols:
    oe = OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)
    X[cat_cols] = oe.fit_transform(X[cat_cols])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=SEED)
print(f"Train: {X_train.shape}, Test: {X_test.shape}")
print(f"Target — Train mean: {y_train.mean():,.2f}, Test mean: {y_test.mean():,.2f}")

## 16 · Preprocessing Strategy

- **Missing values**: None.
- **Categorical**: `crop_type`, `irrigation`, `pesticide_usage` encoded with OrdinalEncoder.
- **Scaling**: Not needed for tree models.
- **Outliers**: Very high yields are from optimal conditions — keep them.

## 17 · Feature Engineering

Sanitize column names and add engineered features.

In [ ]:
X_train = X_train.copy()
X_test = X_test.copy()

X_train["temp_deviation"] = np.abs(X_train["avg_temp_c"] - 25)
X_test["temp_deviation"] = np.abs(X_test["avg_temp_c"] - 25)

X_train["ph_deviation"] = np.abs(X_train["soil_ph"] - 6.5)
X_test["ph_deviation"] = np.abs(X_test["soil_ph"] - 6.5)

X_train["total_nutrients"] = X_train["soil_nitrogen"] + X_train["soil_phosphorus"] + X_train["soil_potassium"]
X_test["total_nutrients"] = X_test["soil_nitrogen"] + X_test["soil_phosphorus"] + X_test["soil_potassium"]

clean = [c.replace(" ", "_").replace("-", "_").replace(".", "_") for c in X_train.columns]
X_train.columns = clean
X_test.columns = clean
print(f"Features ({len(clean)}): {clean}")

## 18 · Baseline Model

Linear Regression as our baseline regressor.

In [ ]:
baseline = LinearRegression()
baseline.fit(X_train, y_train)
y_pred_bl = baseline.predict(X_test)

rmse_bl = root_mean_squared_error(y_test, y_pred_bl)
mae_bl = mean_absolute_error(y_test, y_pred_bl)
r2_bl = r2_score(y_test, y_pred_bl)

print("=" * 50)
print("BASELINE: Linear Regression")
print("=" * 50)
print(f"RMSE : {rmse_bl:,.2f}")
print(f"MAE  : {mae_bl:,.2f}")
print(f"R2   : {r2_bl:.4f}")

## 19 · LazyPredict Benchmark

Fit dozens of regressors in one call for a quick ranking.

In [ ]:
from lazypredict.Supervised import LazyRegressor

lazy = LazyRegressor(verbose=0, ignore_warnings=True)
lazy_models, _ = lazy.fit(X_train, X_test, y_train, y_test)

print("LazyPredict — Top 15 Regressors:")
print(lazy_models.head(15).to_string())

## 20 · FLAML AutoML Run

Automated model selection and hyperparameter tuning (60s budget).

In [ ]:
from flaml import AutoML

flaml_automl = AutoML()
flaml_automl.fit(X_train, y_train, task="regression", time_budget=60,
                 estimator_list=["lgbm", "rf", "extra_tree", "catboost"],
                 verbose=0, seed=SEED)
y_pred_flaml = flaml_automl.predict(X_test)
print(f"FLAML best model: {flaml_automl.best_estimator}")
print(f"RMSE: {root_mean_squared_error(y_test, y_pred_flaml):,.2f}")
print(f"R2  : {r2_score(y_test, y_pred_flaml):.4f}")

## 21 · Additional Models: CatBoost, LightGBM, XGBoost

Train the modern tabular model stack. GPU auto-detected with CPU fallback.

In [ ]:
def gpu_cleanup():
    gc.collect()
    try:
        import torch; torch.cuda.empty_cache()
    except Exception:
        pass

results = {}
timings = {}

# CatBoost
try:
    from catboost import CatBoostRegressor
    t0 = time.perf_counter()
    try:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                task_type="GPU", devices="0", verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    except Exception:
        cb = CatBoostRegressor(iterations=300, learning_rate=0.05, depth=6,
                                verbose=0, random_seed=SEED)
        cb.fit(X_train, y_train, eval_set=(X_test, y_test), early_stopping_rounds=30)
    timings["CatBoost"] = time.perf_counter() - t0
    results["CatBoost"] = cb.predict(X_test)
    print(f"CatBoost RMSE: {root_mean_squared_error(y_test, results['CatBoost']):,.2f}  ({timings['CatBoost']:.1f}s)")
except Exception as e:
    print(f"CatBoost failed: {e}")
gpu_cleanup()

# LightGBM
try:
    import lightgbm as lgb
    t0 = time.perf_counter()
    try:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                device="gpu", verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    except Exception:
        lg = lgb.LGBMRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                verbose=-1, n_jobs=-1, random_state=SEED)
        lg.fit(X_train, y_train, eval_set=[(X_test, y_test)],
               callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
    timings["LightGBM"] = time.perf_counter() - t0
    results["LightGBM"] = lg.predict(X_test)
    print(f"LightGBM RMSE: {root_mean_squared_error(y_test, results['LightGBM']):,.2f}  ({timings['LightGBM']:.1f}s)")
except Exception as e:
    print(f"LightGBM failed: {e}")
gpu_cleanup()

# XGBoost
try:
    from xgboost import XGBRegressor
    t0 = time.perf_counter()
    try:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  device="cuda", tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    except Exception:
        xgb_model = XGBRegressor(n_estimators=300, learning_rate=0.05, max_depth=6,
                                  tree_method="hist", verbosity=0,
                                  n_jobs=-1, random_state=SEED)
        xgb_model.fit(X_train, y_train, eval_set=[(X_test, y_test)],
                      early_stopping_rounds=30, verbose=False)
    timings["XGBoost"] = time.perf_counter() - t0
    results["XGBoost"] = xgb_model.predict(X_test)
    print(f"XGBoost RMSE: {root_mean_squared_error(y_test, results['XGBoost']):,.2f}  ({timings['XGBoost']:.1f}s)")
except Exception as e:
    print(f"XGBoost failed: {e}")
gpu_cleanup()

# Add baseline & FLAML
results["Linear Regression"] = y_pred_bl
results["FLAML"] = y_pred_flaml

## 22 · Top 3 Model Selection

Rank all models by RMSE and select the top 3.

In [ ]:
model_scores = {}
for name, yp in results.items():
    model_scores[name] = {
        "RMSE": round(root_mean_squared_error(y_test, yp), 2),
        "MAE": round(mean_absolute_error(y_test, yp), 2),
        "R2": round(r2_score(y_test, yp), 4),
    }

scores_df = pd.DataFrame(model_scores).T.sort_values("RMSE")
print("All Model Rankings (by RMSE):")
print(scores_df.to_string())

top3_names = scores_df.index[:3].tolist()
print(f"\nTop 3 models: {top3_names}")

## 23 · Final Training and Evaluation of Top 3

Detailed metrics and predicted-vs-actual plots.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for i, name in enumerate(top3_names):
    yp = results[name]
    rmse = root_mean_squared_error(y_test, yp)
    r2 = r2_score(y_test, yp)

    axes[i].scatter(y_test, yp, alpha=0.6, s=40, edgecolors="black")
    mn, mx = min(y_test.min(), yp.min()), max(y_test.max(), yp.max())
    axes[i].plot([mn, mx], [mn, mx], "r--", lw=2)
    axes[i].set_title(f"{name}\nRMSE={rmse:,.2f}  R2={r2:.4f}")
    axes[i].set_xlabel("Actual")
    axes[i].set_ylabel("Predicted")

plt.suptitle("Top 3 Models — Predicted vs Actual", fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "top3_predicted_vs_actual.png"), dpi=120, bbox_inches="tight")
plt.show()

print("\nDetailed Metrics — Top 3:")
for name in top3_names:
    yp = results[name]
    print(f"\n  {name}:")
    print(f"    RMSE : {root_mean_squared_error(y_test, yp):,.2f}")
    print(f"    MAE  : {mean_absolute_error(y_test, yp):,.2f}")
    print(f"    R2   : {r2_score(y_test, yp):.4f}")

## 24 · Error Analysis

Examine residuals from the best model.

In [ ]:
best_name = top3_names[0]
best_pred = results[best_name]
residuals = y_test.values - best_pred

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].hist(residuals, bins=20, color="steelblue", edgecolor="black", alpha=0.8)
axes[0].axvline(0, color="red", linestyle="--")
axes[0].set_title(f"{best_name} — Residual Distribution")
axes[0].set_xlabel("Residual (Actual - Predicted)")

axes[1].scatter(best_pred, residuals, alpha=0.6, s=40, edgecolors="black")
axes[1].axhline(0, color="red", linestyle="--")
axes[1].set_title(f"{best_name} — Residual vs Predicted")
axes[1].set_xlabel("Predicted"); axes[1].set_ylabel("Residual")

sorted_idx = np.argsort(best_pred)
axes[2].scatter(best_pred[sorted_idx], np.abs(residuals[sorted_idx]), alpha=0.6, s=40, edgecolors="black")
axes[2].set_title(f"{best_name} — |Residual| vs Predicted")
axes[2].set_xlabel("Predicted"); axes[2].set_ylabel("|Residual|")

plt.tight_layout()
plt.show()

print(f"Residual stats ({best_name}):")
print(f"  Mean: {residuals.mean():,.2f}, Std: {residuals.std():,.2f}")
print(f"  Min: {residuals.min():,.2f}, Max: {residuals.max():,.2f}")
print(f"  Median: {np.median(residuals):,.2f}")

## 25 · Interpretation and Business Insight

**Key findings:**
- **Crop type** sets the baseline yield (Corn: 5.5 > Rice: 4.5 > Wheat: 3.5 > Soybean: 2.8).
- **Temperature** has an optimal range (~25°C) — deviations reduce yield.
- **Rainfall** increases yield up to ~900mm, then diminishes.
- **Drip irrigation** outperforms all other methods (+15% vs rainfed).
- **Soil pH** near 6.5 is optimal for most crops.
- **Nitrogen** is the most important soil nutrient.

**Business takeaway:** Drip irrigation + optimal fertilization provide the highest yield ROI.

## 26 · Limitations

1. Synthetic data with simplified agronomy models.
2. No pest/disease outbreak data.
3. Missing soil moisture and drainage.
4. No temporal/seasonal patterns.
5. Simplified climate interactions.

## 27 · How to Improve This Project

1. Use real agricultural survey or satellite-derived yields.
2. Add satellite vegetation indices (NDVI).
3. Include soil moisture sensors and weather station data.
4. Model crop growth stages.
5. Add historical yield trends.

## 28 · Production Considerations

- Deploy for farm-level yield forecasting.
- Integrate with precision agriculture platforms.
- Monitor for climate trend impacts.
- Update with seasonal weather data.
- Output yield probability distributions for crop insurance.

## 29 · Common Mistakes

1. Treating temperature as linearly related to yield (it's optimal-range).
2. Ignoring soil pH as a yield factor.
3. Not separating crop types (different baseline yields).
4. Using total fertilizer without diminishing returns.
5. Not accounting for irrigation efficiency differences.

## 30 · Mini Challenge / Exercises

1. Plot yield vs. temperature — confirm the inverted-U shape.
2. Remove `crop_type` — how much does RMSE increase?
3. Create `rainfall_per_temp = rainfall / avg_temp` as a moisture index.
4. Build separate models by crop type.
5. Calculate optimal fertilizer rate (diminishing returns curve).

## 31 · Final Summary / Key Takeaways

1. **Crop type** determines baseline yield potential.
2. **Temperature** has an optimal range (~25°C for most crops).
3. **Rainfall** has diminishing returns beyond ~900mm.
4. **Drip irrigation** provides the highest yield boost.
5. **Soil nutrients** (N > P > K) are key controllable factors.
6. **Precision agriculture** features improve predictions significantly.

## Save Metrics

In [ ]:
metrics_out = {}
for name, yp in results.items():
    metrics_out[name] = {
        "rmse": round(float(root_mean_squared_error(y_test, yp)), 2),
        "mae": round(float(mean_absolute_error(y_test, yp)), 2),
        "r2": round(float(r2_score(y_test, yp)), 4),
    }

metrics_path = os.path.join(SAVE_DIR, "metrics.json")
with open(metrics_path, "w") as f:
    json.dump(metrics_out, f, indent=2)
print(f"Metrics saved to {metrics_path}")
print(json.dumps(metrics_out, indent=2))